In [2]:
import numpy as np
import jax
import jax.numpy as jnp
import util.functions as functions
from models.FNO import CAPE_FNO
import pybamm
import flax

In [3]:
from util.FNO_util import preprocess_data, train_test_split, remove_padding, normalise_diffusion
from util.plotting import create_plot_FNO_results, create_plot_voltage

In [ ]:
family = "CC"
N_total = 33000
data = np.load(f"../data/{family}_{N_total}.npz")
random_seed = 42
np.random.seed(random_seed)
np.random.randint(0, N_total)

In [ ]:
func_I = np.array(data["current"][0])

### Anode data ###
cn_anode = np.array(data["cn_anode"][0])
c0_anode = np.array(data["c0_anode"][0])
D_anode = np.array(data["Dan"][0])

### Cathode data ###
cn_cathode = np.array(data["cn_cathode"][0])
c0_cathode = np.array(data["c0_cathode"][0])
D_cathode = np.array(data["Dca"][0])

In [ ]:
D_anode = normalise_diffusion(D_anode)
D_cathode = normalise_diffusion(D_cathode)

In [ ]:
# Padding amounts
padding_t = 5  # along t-axis
padding_r = 2  # along r-axis

# Original sample counts
num_samples_I = 75
num_samples_c0 = 20

In [ ]:
X_test_anode, Y_test_anode = preprocess_data(test_I, test_c0_anode, test_cn_anode, num_samples_I, num_samples_c0, padding_r, padding_t)

In [ ]:
# Assume these hyperparameters
k_modes = 10
fno_depth = 8
hidden_channels = 64
input_channels = 4 # We should automate this TODO
output_channels = 1
cape_hidden_size = 32

In [ ]:
model = CAPE_FNO(k_modes=k_modes, input_channels= input_channels, 
                 fno_depth=fno_depth, cape_hidden_size = cape_hidden_size, 
                 hidden_channels=hidden_channels, output_channels=output_channels)

main_key = jax.random.PRNGKey(random_seed)
# Initialize parameters
dummy_D = jax.random.normal(main_key, (1,1))
params = model.init(main_key, X_train_anode[:1,...], dummy_D)

# Forward pass
out = model.apply(params, X_train_anode[:1,...], dummy_D)

In [ ]:
anode_file = "../trained_models/cape_fno/anode_CC_2025-05-29_11-07-33.msgpack"
cathode_file = "../trained_models/cape_fno/cathode_CC_2025-05-29_11-31-53.msgpack"

params_anode = functions.load_model_params(anode_file)
params_cathode = functions.load_model_params(cathode_file)

params_anode = flax.serialization.from_bytes(params, params_anode)
params_cathode = flax.serialization.from_bytes(params, params_cathode)